In [0]:
from pyspark.sql.functions import *

customers_df = spark.table("workspace.default.silver_customers")

products_df = spark.table("workspace.default.silver_products")

orders_df = spark.table("workspace.default.silver_orders")

payments_df = spark.table("workspace.default.silver_payments")

shipments_df = spark.table("workspace.default.silver_shipments")

In [0]:
print("Customers :", customers_df.count())
print("Products  :", products_df.count())
print("Orders    :", orders_df.count())
print("Payments  :", payments_df.count())
print("Shipments :", shipments_df.count())

In [0]:
sales_df = orders_df.join(
    customers_df,
    on="customer_id",
    how="left"
)

sales_df.show(5)

In [0]:
sales_df = sales_df.join(
    products_df,
    on="product_id",
    how="left"
)

sales_df.show(5)

In [0]:
sales_df.printSchema()

In [0]:
print("Total Records :", sales_df.count())

In [0]:
sales_df.show(5)

sales_df.printSchema()

print(sales_df.count())

In [0]:
sales_df = sales_df.join(
    payments_df,
    on="order_id",
    how="left"
)

sales_df.show(5)

In [0]:
print("Orders Count :", orders_df.count())

print("Sales Count :", sales_df.count())

In [0]:
payments_df.groupBy("order_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

In [0]:
sales_df = sales_df.join(
    shipments_df,
    on="order_id",
    how="left"
)

sales_df.show(5)

In [0]:
print("Orders Count :", orders_df.count())

print("Sales Count :", sales_df.count())

In [0]:
sales_df.printSchema()

In [0]:
from pyspark.sql.functions import col

sales_df = sales_df.withColumn(
    "total_amount",
    col("price") * col("quantity")
)

In [0]:
sales_df.select(
    "order_id",
    "price",
    "quantity",
    "total_amount"
).show(10)

In [0]:
from pyspark.sql.functions import when

sales_df = sales_df.withColumn(
    "payment_flag",
    when(col("payment_status") == "SUCCESS", "Paid")
    .otherwise("Not Paid")
)

In [0]:
sales_df.select(
    "payment_status",
    "payment_flag"
).show()

In [0]:
sales_df = sales_df.withColumn(
    "delivery_flag",
    when(col("shipment_status") == "Delivered", "Delivered")
    .otherwise("Pending")
)

In [0]:
sales_df.select(
    "shipment_status",
    "delivery_flag"
).show()

In [0]:
sales_df.select(
    "order_id",
    "price",
    "quantity",
    "total_amount",
    "payment_status",
    "payment_flag",
    "shipment_status",
    "delivery_flag"
).show(10, False)

In [0]:
from pyspark.sql.functions import when, col

sales_df = sales_df.withColumn(
    "order_status",
    when(
        (col("payment_status") == "SUCCESS") &
        (col("shipment_status") == "Delivered"),
        "COMPLETED"
    )
    .when(
        col("payment_status") == "FAILED",
        "PAYMENT_FAILED"
    )
    .otherwise("IN_PROGRESS")
)

In [0]:
sales_df.select(
    "payment_status",
    "shipment_status",
    "order_status"
).show(20, False)

In [0]:
sales_df = sales_df.select(

    "order_id",
    "customer_id",
    "customer_name",
    "email",
    "city",

    "product_id",
    "product_name",
    "category",
    "brand",

    "price",
    "quantity",
    "total_amount",

    "payment_method",
    "payment_status",
    "payment_flag",

    "shipment_status",
    "delivery_flag",

    "order_status",

    "order_date"
)

In [0]:
sales_df.show(10, False)

sales_df.printSchema()

print("Total Records :", sales_df.count())

In [0]:
from pyspark.sql.functions import current_timestamp, lit

sales_df = (
    sales_df
    .withColumn("load_timestamp", current_timestamp())
    .withColumn("pipeline_name", lit("Silver_Order_Transformation"))
)

In [0]:
sales_df.select(
    "order_id",
    "load_timestamp",
    "pipeline_name"
).show(5,False)

In [0]:
sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_sales")

In [0]:
spark.sql("""
SELECT *
FROM workspace.default.silver_sales
LIMIT 10
""").show(truncate=False)

In [0]:
print("Silver Sales Count :", sales_df.count())

print(
    "Table Count :",
    spark.table("workspace.default.silver_sales").count()
)